# Import

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import gmsh
from pathlib import Path
import sys
import meshio

%load_ext autoreload
%autoreload 2

notebook_dir = Path("/home/jlangbehn/documents/private/generative_art/chladni_eigenmodes")
sys.path.append(str(notebook_dir))

from chladni_eigenmodes.fem_utils import meshio_to_points_triangles, hdf5_filename_for_source, save_fem_solution_hdf5, plot_mesh
from chladni_eigenmodes.text_to_gmsh import text_to_shapely

# Generative Art

In [23]:
sly = text_to_shapely("Generative Art", x_scale=0.9, underline=True,
                      font="DejaVu Sans Mono",
                      #font="FreeMono",
                      underline_height=0.1
                      )

display(sly)
if sly.geom_type == "Polygon":
    geoms = [sly]
else:
    geoms = sly.geoms

fig, ax = plt.subplots()
for geom in geoms:
    ax.plot(geom.exterior.xy[0], geom.exterior.xy[1])

    for interior in geom.interiors:
        ax.plot(interior.xy[0], interior.xy[1])

ax.set_aspect("equal")
plt.show()

## Single Polygon

In [24]:
import gmsh

gmsh.initialize()
model = gmsh.model
occ = model.occ

outer_lc = 0.05
coords = list(sly.exterior.coords)

# Shapely exterior.coords usually repeats the first point at the end.
# Keep it closed for line construction, but do not create duplicate points unnecessarily.
if coords[0] != coords[-1]:
    coords.append(coords[0])

outer_pts = [
    occ.addPoint(x, y, 0.0, outer_lc)
    for x, y in coords[:-1]
]

outer_lines = []
n = len(outer_pts)

for i in range(n):
    p1 = outer_pts[i]
    p2 = outer_pts[(i + 1) % n]
    outer_lines.append(occ.addLine(p1, p2))

outer_loop = occ.addCurveLoop(outer_lines)
inner_loops = []
for interior in sly.interiors:
    coords = list(interior.coords)
    if coords[0] != coords[-1]:
        coords.append(coords[0])
    inner_pts = [
        occ.addPoint(x, y, 0.0, outer_lc)
        for x, y in coords[:-1]
    ]
    inner_lines = []
    n = len(inner_pts)
    for i in range(n):
        p1 = inner_pts[i]
        p2 = inner_pts[(i + 1) % n]
        inner_lines.append(occ.addLine(p1, p2))
    inner_loop = occ.addCurveLoop(inner_lines)
    inner_loops.append(inner_loop)

surface = occ.addPlaneSurface([outer_loop] + inner_loops)

occ.synchronize()

# Mesh settings
#gmsh.option.setNumber("Mesh.CharacteristicLengthMin", mesh_char_length)
#gmsh.option.setNumber("Mesh.CharacteristicLengthMax", mesh_char_length)

# Generate mesh
gmsh.model.mesh.generate(2)

filename = "/home/jlangbehn/documents/private/generative_art/chladni_eigenmodes/output/test.msh"
gmsh.write(filename)
gmsh.finalize()

# Read mesh with meshio
mesh = meshio.read(filename)
plot_mesh(mesh)

## Multi Polygon

In [29]:
from shapely.geometry import Polygon, MultiPolygon
import gmsh


def make_curve_loop_from_coords(coords, lc):
    """
    Create a polygonal Gmsh curve loop from a coordinate sequence.
    The sequence may or may not be closed.
    """
    occ = gmsh.model.occ

    coords = list(coords)

    # Remove repeated final point for point creation
    if coords[0] == coords[-1]:
        coords = coords[:-1]

    pts = [
        occ.addPoint(float(x), float(y), 0.0, lc)
        for x, y in coords
    ]

    lines = []
    n = len(pts)

    for i in range(n):
        p1 = pts[i]
        p2 = pts[(i + 1) % n]
        lines.append(occ.addLine(p1, p2))

    return occ.addCurveLoop(lines)


def add_polygon_surface(poly, lc):
    """
    Add a single Shapely Polygon, including holes, as a Gmsh plane surface.
    Returns the surface tag.
    """
    occ = gmsh.model.occ

    outer_loop = make_curve_loop_from_coords(poly.exterior.coords, lc)

    hole_loops = [
        make_curve_loop_from_coords(interior.coords, lc)
        for interior in poly.interiors
    ]

    surface = occ.addPlaneSurface([outer_loop] + hole_loops)

    return surface


def add_shapely_geometry_as_surface_group(
    geom,
    lc,
    physical_name="domain",
):
    """
    Add a Shapely Polygon or MultiPolygon to Gmsh.

    Returns:
        surfaces: list of 2D surface tags
        physical_group: physical group tag collecting all surfaces
    """
    if isinstance(geom, Polygon):
        polygons = [geom]
    elif isinstance(geom, MultiPolygon):
        polygons = list(geom.geoms)
    else:
        raise TypeError(f"Expected Polygon or MultiPolygon, got {type(geom)}")

    surfaces = [
        add_polygon_surface(poly, lc)
        for poly in polygons
    ]

    gmsh.model.occ.synchronize()

    physical_group = gmsh.model.addPhysicalGroup(2, surfaces)
    gmsh.model.setPhysicalName(2, physical_group, physical_name)

    return surfaces, physical_group

gmsh.initialize()
gmsh.model.add("multipolygon_example")

lc = 0.05

surfaces, domain_tag = add_shapely_geometry_as_surface_group(
    sly,
    lc=lc,
    physical_name="my_multipolygon",
)

mesh_char_length = 0.01
gmsh.option.setNumber("Mesh.CharacteristicLengthMin", mesh_char_length)
gmsh.option.setNumber("Mesh.CharacteristicLengthMax", mesh_char_length)

gmsh.model.mesh.generate(2)


filename = "/home/jlangbehn/documents/private/generative_art/chladni_eigenmodes/output/generative_art.msh"
gmsh.write(filename)
gmsh.finalize()

# Read mesh with meshio
mesh = meshio.read(filename)
plot_mesh(mesh)